In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd "/content/drive/MyDrive/Colab Notebooks/cs1851"

/content/drive/MyDrive/Colab Notebooks/cs1851


In [3]:

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, Subset
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from random_forest_pipeline import load_training_data, fit_preprocessors, transform_with_preprocessors, make_train_val_split, build_random_forest, tune_random_forest, evaluate_model
# from cnn_pipeline import
from cnn_pipeline import CNN, FCN, train_one_epoch, run_experiment, evaluate, ImageDataset, build_dataloader


## Data + Setup

In [4]:
# generic path
path = "/content/drive/MyDrive/"

In [5]:
# edit data_path to point to data
data_path = "Colab Notebooks/cs1851/data/"
images = np.load(path + data_path + "train_images.npy")
labels = np.load(path + data_path + "train_labels.npy")
ids = np.load(path + data_path + "train_ids.npy", allow_pickle=True)

# Image dataset
train_loader, val_loader = build_dataloader(images, labels, ids)

# Tabular data
X_raw, y, ids = load_training_data(data_dir="data")
X_train_raw, X_val_raw, y_train, y_val, train_ids, val_ids = make_train_val_split(
        X_raw, y, ids, test_size=0.2, random_state=42
    )
X_train, age_median, sex_encoder, site_encoder = fit_preprocessors(X_train_raw)
X_val = transform_with_preprocessors(X_val_raw, age_median, sex_encoder, site_encoder)
print("Processed X_train shape:", X_train.shape)
print("Processed X_val shape:", X_val.shape)

Processed X_train shape: (2800, 3)
Processed X_val shape: (700, 3)


In [6]:
torch.manual_seed(42)
np.random.seed(42)

EPOCHS = 100
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu") #this ensures that the code runs on GPUs if available
print(f"Device: {DEVICE}\n")

Device: cuda



## Train Models

### Tabular Model (RF)

In [7]:
rf, best_params, best_score = tune_random_forest(X_train, y_train)
rf.fit(X_train, y_train)
metrics, y_pred, y_proba = evaluate_model(rf, X_val, y_val)

Fitting 3 folds for each of 162 candidates, totalling 486 fits

Best Parameters
   max_depth max_features  min_samples_leaf  min_samples_split  n_estimators
0         10         sqrt                 1                 10           300

Best CV Macro F1
   f1_macro_cv
0       0.2186

Validation Metrics
   accuracy  precision_macro  recall_macro  f1_macro  f1_weighted
0    0.5086           0.2179        0.2362    0.2162       0.5606

Classification Report
              precision    recall  f1-score   support

           0       0.18      0.22      0.20        77
           1       0.18      0.24      0.21        41
           2       0.12      0.33      0.18        24
           3       0.15      0.19      0.17        77
           4       0.89      0.66      0.76       463
           5       0.00      0.00      0.00         8
           6       0.00      0.00      0.00        10

    accuracy                           0.51       700
   macro avg       0.22      0.24      0.22       700
w

In [12]:
val_results = pd.DataFrame({
        "ID": val_ids,
        "true_label": y_val,
        "rf_pred": y_pred
    })
for c in range(y_proba.shape[1]):
    val_results[f"rf_prob_{c}"] = y_proba[:, c]

# saving results to csv (for now)
# change file_path to specific location
file_path = "Colab Notebooks/cs1851/rf_val_predictions.csv"
rf_save_path = os.path.join(os.path.dirname(path), file_path)
val_results.to_csv(rf_save_path, index=False)
print(f"\nSaved validation probabilities to: {rf_save_path}")

### Image Models (CNN & FCN)

CNN

In [7]:
num_classes = 7

cnn = CNN(device=DEVICE, train_loader=train_loader, val_loader=val_loader, num_classes=num_classes).to(DEVICE)
hist_base, probs, preds, labels, ids  = run_experiment(cnn, EPOCHS, lr=1e-4)

  Epoch  10 | train loss 0.9505  | train acc 38.7292
val loss 0.9299
  Epoch  20 | train loss 0.8454  | train acc 40.5833
val loss 0.8733
  Epoch  30 | train loss 0.7382  | train acc 42.6458
val loss 0.8604
  Epoch  40 | train loss 0.6063  | train acc 44.9167
val loss 0.8925
  Epoch  50 | train loss 0.4475  | train acc 49.3750
val loss 0.9742
  Epoch  60 | train loss 0.3496  | train acc 51.2500
val loss 1.1146
  Epoch  70 | train loss 0.2287  | train acc 53.7500
val loss 1.3007
  Epoch  80 | train loss 0.1529  | train acc 55.0208
val loss 1.6003
  Epoch  90 | train loss 0.1244  | train acc 55.7708
val loss 1.6812
  Epoch 100 | train loss 0.0870  | train acc 56.6875
val loss 1.9060


In [8]:
val_results = pd.DataFrame({
        "ID": ids,
        "true_label": labels,
        "cnn_pred": preds
    })
for c in range(probs.shape[1]):
    val_results[f"cnn_prob_{c}"] = probs[:, c]

file_path = "Colab Notebooks/cs1851/cnn_val_predictions.csv"
cnn_save_path = os.path.join(os.path.dirname(path), file_path)
val_results.to_csv(cnn_save_path, index=False)
print(f"\nSaved validation probabilities to: {cnn_save_path}")


Saved validation probabilities to: /content/drive/MyDrive/Colab Notebooks/cs1851/cnn_val_predictions.csv


FCN

In [9]:
criterion = nn.CrossEntropyLoss()

fcn = FCN(criterion, train_loader, val_loader, DEVICE, lr=1e-3).to(DEVICE)
hist_fully, _ ,val_loss, probs, preds, labels, ids = fcn.fit(epochs=EPOCHS)

Epoch 10 loss: 0.9378535815647671 val loss 1.006931678227016
Epoch 20 loss: 0.9132648951666695 val loss 0.917986981528146
Epoch 30 loss: 0.8832471820286342 val loss 0.9328027514048985
Epoch 40 loss: 0.8583925761495318 val loss 0.9596978391919817
Epoch 50 loss: 0.8469250896998815 val loss 0.9300201327460152
Epoch 60 loss: 0.810165605204446 val loss 0.9110370349884033
Epoch 70 loss: 0.7688422730990818 val loss 0.9078496210915702
Epoch 80 loss: 0.7255132978303092 val loss 0.9071518346241543
Epoch 90 loss: 0.6915364180292402 val loss 0.9241845968791417
Epoch 100 loss: 0.6730235859325954 val loss 0.9302720165252686
Epoch 100/100  |  Loss = 42.8288


In [10]:
val_results = pd.DataFrame({
        "ID": ids,
        "true_label": labels,
        "fcn_pred": preds
    })
for c in range(probs.shape[1]):
    val_results[f"fcn_prob_{c}"] = probs[:, c]

file_path = "Colab Notebooks/cs1851/fcn_val_predictions.csv"
fcn_save_path = os.path.join(os.path.dirname(path), file_path)
val_results.to_csv(fcn_save_path, index=False)
print(f"\nSaved validation probabilities to: {fcn_save_path}")


Saved validation probabilities to: /content/drive/MyDrive/Colab Notebooks/cs1851/fcn_val_predictions.csv


In [13]:
rf_df = pd.read_csv(rf_save_path)
print(rf_df.columns)
cnn_df = pd.read_csv(cnn_save_path)
print(cnn_df.columns)
fcn_df = pd.read_csv(fcn_save_path)
print(fcn_df.columns)

Index(['ID', 'true_label', 'rf_pred', 'rf_prob_0', 'rf_prob_1', 'rf_prob_2',
       'rf_prob_3', 'rf_prob_4', 'rf_prob_5', 'rf_prob_6'],
      dtype='object')
Index(['ID', 'true_label', 'cnn_pred', 'cnn_prob_0', 'cnn_prob_1',
       'cnn_prob_2', 'cnn_prob_3', 'cnn_prob_4', 'cnn_prob_5', 'cnn_prob_6'],
      dtype='object')
Index(['ID', 'true_label', 'fcn_pred', 'fcn_prob_0', 'fcn_prob_1',
       'fcn_prob_2', 'fcn_prob_3', 'fcn_prob_4', 'fcn_prob_5', 'fcn_prob_6'],
      dtype='object')


In [14]:
# print(rf_y_val.tolist() == cnn_y_val.tolist())

def get_probs(df, model_type):
    probs = df[[f'{model_type}_prob_{i}' for i in range(7)]].values
    return probs

def avg_fusion(tab_df, img_df, tab_model, img_model):

    merged = pd.merge(tab_df, img_df, on="ID")
    # print(merged.columns)
    tab_probs = get_probs(merged, tab_model)
    img_probs = get_probs(merged, img_model)

    y = merged['true_label_x'].values

    fusion_probs = 0.5 * tab_probs + 0.5 * img_probs
    fusion_preds = fusion_probs.argmax(axis=1)
    return accuracy_score(y, fusion_preds)

print("Averaged Late Fusion")
print("RF + CNN accuracy:", avg_fusion(rf_df, cnn_df, 'rf', 'cnn'))
print("RF + FCN accuracy:", avg_fusion(rf_df, fcn_df, 'rf', 'fcn'))
# cnn 1 epoch fused prob: 0.6557142857142857

Averaged Late Fusion
RF + CNN accuracy: 0.7071428571428572
RF + FCN accuracy: 0.7142857142857143


In [15]:
# fusion features
from sklearn.linear_model import LogisticRegression
def lr_fusion(tab_df, img_df, tab_model, img_model):
    merged = pd.merge(tab_df, img_df, on="ID")
    # print(merged.columns)
    tab_probs, img_probs = get_probs(tab_df, tab_model), get_probs(img_df, img_model)

    features = np.concatenate([tab_probs, img_probs], axis=1)
    lr = LogisticRegression(class_weight='balanced', max_iter=3000)
    y = merged['true_label_x']
    lr.fit(features, y)

    lr_fusion_probs = lr.predict_proba(features)
    fusion_preds = lr_fusion_probs.argmax(axis=1)
    return accuracy_score(y, fusion_preds)

print("Logistic Regression Late Fusion")
print("RF + CNN accuracy", lr_fusion(rf_df, cnn_df, 'rf', 'cnn'))
print("RF + FCN accuracy", lr_fusion(rf_df, fcn_df, 'rf', 'fcn'))

# rf + CNN 1 epoch
# 0.3942857142857143



Logistic Regression Late Fusion
RF + CNN accuracy 0.6357142857142857
RF + FCN accuracy 0.6557142857142857
